#### DATA QUALITY SCORE


%md
##### 10.1 DOCUMENTAÇÃO DO DATA QUALITY SCORE

###### Objetivo

O **Data Quality Score** é responsável por consolidar indicadores produzidos pelos diferentes Profiles do framework em uma avaliação quantitativa, estruturada e interpretável da qualidade dos dados.

O objetivo é transformar métricas individuais em indicadores consolidados, permitindo avaliar o comportamento das dimensões de qualidade, identificar atributos que merecem atenção e acompanhar a evolução da qualidade dos datasets.

O módulo foi desenvolvido utilizando **PySpark**, com processamento distribuído e funções reutilizáveis disponibilizadas na biblioteca `00_FRAMEWORK_CONFIG`.

Sua arquitetura é **genérica, configurável e independente da origem dos dados**, podendo ser aplicada a qualquer DataFrame Spark compatível com os contratos de entrada.

A análise busca responder:

> Qual é o nível de qualidade observado no dataset, considerando os critérios efetivamente avaliados, e qual é a cobertura dessa avaliação?

O Data Quality Score não pressupõe que todas as métricas exploratórias representem problemas de qualidade.

---

###### Princípios fundamentais

A construção do Score considera cinco princípios:

**1. Independência da fonte**

O cálculo utiliza resultados padronizados dos Profiles, sem depender de arquivos CSV, tabelas Delta, bancos relacionais, APIs ou qualquer outra origem física.

**2. Separação entre evidência e penalização**

Nem toda característica estatística identificada representa um defeito.

Correlação elevada, valores extremos, padrões minoritários e concentrações de frequência podem ser comportamentos legítimos.

**3. Elegibilidade explícita**

Somente métricas habilitadas e tecnicamente válidas participam do cálculo.

Indicadores não aplicáveis, inválidos ou exclusivamente exploratórios não devem receber pontuação artificial.

**4. Transparência dos pesos**

Os pesos utilizados na consolidação são configuráveis e devem permanecer disponíveis para auditoria e interpretação.

**5. Cobertura da avaliação**

O Score deve ser apresentado juntamente com informações sobre as métricas avaliadas, evitando interpretações equivocadas de resultados baseados em cobertura limitada.

---

###### Exemplos

**Exemplo 1 — Score de completude**

Considere um DataFrame com três colunas:

| Coluna | Preenchimento |
|---|---:|
| coluna_a | 100% |
| coluna_b | 90% |
| coluna_c | 80% |

Com a completude habilitada e pesos iguais, o Score operacional será:

`Score = (100 + 90 + 80) / 3 = 90`

O resultado indica uma completude média de 90% entre as colunas avaliadas.

Entretanto, esse indicador não comprova que os valores ausentes representam erros, pois a obrigatoriedade dos atributos depende de regras específicas.

**Exemplo 2 — Consolidação ponderada**

Considere duas métricas elegíveis:

| Métrica | Score | Peso |
|---|---:|---:|
| Completude | 90 | 2 |
| Regra temporal explícita | 80 | 1 |

O Score consolidado será:

`Score = ((90 × 2) + (80 × 1)) / (2 + 1)`

`Score = 86,67`

Nesse exemplo, a completude possui maior influência devido ao peso configurado.

**Exemplo 3 — Métrica não aplicável**

Considere um dataset que não possui regras temporais configuradas.

Nesse cenário, a dimensão temporal não deve receber automaticamente Score igual a zero ou cem.

A métrica permanece fora do cálculo até que exista uma regra elegível e avaliável.

**Exemplo 4 — Evidência exploratória**

Considere duas variáveis com correlação de Pearson igual a `0,98`.

Esse resultado representa uma associação linear elevada, mas não comprova baixa qualidade.

Portanto, a correlação pode ser registrada como evidência exploratória sem reduzir o Score.

---

###### Principais análises

O Data Quality Score contempla:

1. **Validação dos insumos**
   - Verificação da existência dos DataFrames necessários;
   - Validação das colunas obrigatórias;
   - Verificação das funções disponíveis na biblioteca;
   - Aplicação de contratos padronizados.

2. **Configuração dos critérios**
   - Habilitação ou desabilitação de métricas;
   - Definição de pesos;
   - Exclusão configurável de colunas;
   - Definição de regras explicitamente avaliáveis.

3. **Normalização dos indicadores**
   - Conversão de métricas para uma escala de 0 a 100;
   - Validação dos intervalos permitidos;
   - Tratamento de valores NULL, NaN e infinitos;
   - Preservação de indicadores não aplicáveis.

4. **Elegibilidade das métricas**
   - Identificação dos critérios habilitados;
   - Diferenciação entre métricas pontuáveis e exploratórias;
   - Exclusão de indicadores sem validade técnica;
   - Registro da origem e da justificativa de cada métrica.

5. **Score por dimensão**
   - Consolidação de métricas elegíveis;
   - Aplicação dos pesos configurados;
   - Cálculo da pontuação por dimensão;
   - Identificação de dimensões sem avaliação aplicável.

6. **Score por coluna**
   - Consolidação dos indicadores por atributo;
   - Identificação de colunas com menor pontuação;
   - Apoio à priorização de investigações.

7. **Score geral do dataset**
   - Cálculo da média ponderada dos critérios elegíveis;
   - Exclusão de métricas não aplicáveis;
   - Apresentação da pontuação consolidada;
   - Registro da quantidade de métricas efetivamente pontuadas.

8. **Cobertura da avaliação**
   - Quantidade de métricas catalogadas;
   - Quantidade de métricas pontuadas;
   - Percentual de cobertura do catálogo;
   - Identificação das dimensões não pontuadas.

9. **Consolidação de evidências exploratórias**
   - Identificação dos Profiles que produzem sinais diagnósticos;
   - Preservação de métricas não pontuáveis;
   - Separação entre observações estatísticas e critérios de qualidade.

10. **Validação dos resultados**
    - Verificação da fórmula de ponderação;
    - Testes de elegibilidade;
    - Tratamento de ausência de métricas;
    - Verificação dos limites do Score.

---

###### Metodologia de cálculo

O Score utiliza uma média ponderada das métricas elegíveis.

A fórmula geral é:

$$
DQScore =
\frac{\sum_{i=1}^{n} w_i \cdot s_i}
{\sum_{i=1}^{n} w_i}
$$

Onde:

- `DQScore`: pontuação consolidada;
- `s_i`: pontuação normalizada da métrica;
- `w_i`: peso configurado;
- `n`: quantidade de métricas elegíveis.

Somente métricas habilitadas, válidas e com peso positivo participam do cálculo.

O resultado é apresentado na escala de **0 a 100**.

Quando não existem métricas elegíveis, o Score deve permanecer **NULL**, indicando ausência de avaliação aplicável.

---

###### Cobertura da avaliação

A cobertura representa a proporção de métricas catalogadas que efetivamente participaram do cálculo.

$$
Cobertura =
\frac{\text{Métricas pontuadas}}
{\text{Métricas catalogadas}}
\times 100
$$

Esse indicador permite distinguir a pontuação obtida da abrangência da avaliação.

**Importante:** a cobertura do catálogo não equivale necessariamente à cobertura de registros, colunas ou dimensões do dataset.

Por exemplo, um Score de 95 com cobertura limitada não deve ser interpretado como garantia de que 95% dos dados estão corretos.

---

###### Integração com os Profiles

O Data Quality Score considera as evidências produzidas pelos módulos anteriores.

| Profile | Contribuição | Tratamento inicial |
|---|---|---|
| 03 — Schema | Completude e características estruturais | Score operacional configurável |
| 04 — Distribution | Concentração, diversidade e regularidade | Evidência exploratória |
| 05 — Pattern | Estruturas e desvios de padrão | Evidência exploratória |
| 06 — Duplicity | Duplicidades e unicidade | Evidência exploratória |
| 07 — Consistency | Consistência intrínseca e regras explícitas | Regras temporais opcionais |
| 08 — Outlier | Valores estatisticamente extremos | Evidência exploratória |
| 09 — Correlation | Associações entre variáveis | Evidência exploratória |

A participação de uma métrica no Score depende de sua elegibilidade e configuração.

A ausência de pontuação automática não significa que a dimensão seja irrelevante.

Significa apenas que suas evidências não devem ser convertidas em penalizações sem critérios adequados.

---

###### Tratamento de situações especiais

O módulo considera:

- **NULL:** não recebe pontuação artificial;
- **NaN:** tratado como indicador inválido;
- **Valores infinitos:** excluídos da normalização;
- **Percentuais fora de 0 a 100:** considerados inválidos;
- **Pesos não positivos:** não contribuem para o cálculo;
- **Métricas desabilitadas:** permanecem fora do Score;
- **Dimensões sem métricas elegíveis:** não recebem pontuação;
- **Datasets vazios:** exigem interpretação específica dos indicadores disponíveis;
- **Ausência de regras explícitas:** não representa automaticamente qualidade perfeita ou inadequada.

O tratamento dessas situações evita distorções na avaliação consolidada.

---

###### Interpretação dos resultados

O Score representa uma síntese quantitativa dos critérios avaliados.

Uma pontuação elevada pode indicar bom desempenho nas métricas habilitadas, mas não garante ausência de problemas.

Uma pontuação reduzida indica que determinadas métricas avaliadas apresentaram resultados inferiores, exigindo investigação.

A interpretação deve considerar:

- Critérios habilitados;
- Pesos configurados;
- Cobertura da avaliação;
- Quantidade de métricas pontuadas;
- Natureza dos indicadores;
- Ausência de regras específicas de negócio.

**Princípio fundamental:**

> O Data Quality Score representa a qualidade observada segundo os critérios avaliados, e não uma certificação absoluta da qualidade do dataset.

O framework não estabelece automaticamente um percentual universal de aprovação, pois limites aceitáveis dependem dos requisitos e do contexto de utilização dos dados.

---

###### Características do módulo

O Data Quality Score apresenta as seguintes características:

- **Genérico:** independente de regras específicas de domínio;
- **Modular:** integra os Profiles existentes;
- **Reutilizável:** aplicável a diferentes DataFrames;
- **Configurável:** permite ajustar métricas e pesos;
- **Escalável:** utiliza operações distribuídas do PySpark;
- **Auditável:** mantém origem, natureza e justificativa das métricas;
- **Explicável:** disponibiliza os componentes do cálculo;
- **Não destrutivo:** não modifica os registros da fonte;
- **Extensível:** permite incorporar novos indicadores;
- **Conservador:** não transforma automaticamente evidências exploratórias em defeitos.

As funções genéricas de normalização, validação e agregação são mantidas em `00_FRAMEWORK_CONFIG`.

O notebook `10_DATA_QUALITY_SCORE` concentra a configuração, a integração dos Profiles, a consolidação e a apresentação dos resultados.

---

###### DataFrames de resultado

O módulo disponibiliza cinco DataFrames principais.

**`dq_score_metrics_df`**

Contém o catálogo de métricas, incluindo dimensão, coluna, valor observado, pontuação, peso, elegibilidade, natureza, origem e justificativa.

**`dq_score_dimension_df`**

Apresenta o Score consolidado por dimensão, acompanhado da quantidade de métricas e da cobertura da avaliação.

**`dq_score_column_df`**

Apresenta o Score por coluna, permitindo identificar atributos que merecem investigação.

**`dq_score_dataset_df`**

Disponibiliza o Score consolidado do dataset, calculado a partir das métricas elegíveis.

**`dq_score_evidence_df`**

Registra a disponibilidade de evidências exploratórias que não participam automaticamente da pontuação.

---

###### Limitações

A avaliação possui limitações que devem ser consideradas:

1. A completude não representa necessariamente obrigatoriedade de preenchimento.
2. A ausência de duplicidades não garante correção dos dados.
3. Valores extremos não representam necessariamente erros.
4. Correlações estatísticas não demonstram causalidade.
5. Padrões minoritários podem representar situações legítimas.
6. O Score depende dos critérios e pesos configurados.
7. A cobertura do catálogo não garante abrangência integral do dataset.
8. Métricas exploratórias exigem interpretação adicional.
9. Um Score elevado não substitui validações específicas de negócio.

Essas limitações devem permanecer explícitas para evitar conclusões indevidas.

---

###### Resultado esperado

Ao final da execução, o Data Quality Score deverá permitir responder:

1. Qual é o Score consolidado do dataset?
2. Quais métricas participaram efetivamente do cálculo?
3. Quais dimensões possuem indicadores pontuáveis?
4. Quais colunas apresentam menores pontuações?
5. Qual é a cobertura da avaliação?
6. Quais métricas foram excluídas do cálculo?
7. Quais evidências permanecem exclusivamente exploratórias?
8. Como os pesos influenciam o resultado?
9. Existem dimensões sem avaliação aplicável?
10. Quais indicadores devem ser priorizados para investigação?

Os resultados poderão alimentar futuras etapas de **orquestração, monitoramento histórico, observabilidade e visualização de qualidade dos dados**.

O Data Quality Score constitui a camada de consolidação quantitativa do framework, preservando a rastreabilidade das métricas e a distinção entre evidências estatísticas e critérios efetivos de qualidade.


In [0]:
%run "./03_SCHEMA_PROFILE"

In [0]:
# ============================================================
# 10.3 CONFIGURAÇÕES
# ============================================================

# O QUE FAZ:
# Define os critérios elegíveis e pesos do Score.

# COMO FAZ:
# Centraliza opções de completude, regras temporais, exclusões e testes.

# POR QUE É IMPORTANTE:
# Evita penalizações implícitas e permite reprodução da avaliação.

# PERGUNTA RESPONDIDA:
# "Quais critérios participam do Score?"


DQ_SCORE_DISPLAY_RESULTS = bool(globals().get("DISPLAY_RESULTS", True))
DQ_SCORE_RUN_TESTS = False

# Completude é um proxy operacional, não uma obrigação universal.
DQ_SCORE_ENABLE_COMPLETENESS = True
DQ_SCORE_USE_EFFECTIVE_COMPLETENESS = False
DQ_SCORE_COMPLETENESS_WEIGHT = 1.0
DQ_SCORE_EXCLUDED_COLUMNS = []

# Regras temporais exigem declaração explícita no módulo 07.
DQ_SCORE_ENABLE_TEMPORAL_RULES = False
DQ_SCORE_TEMPORAL_WEIGHT = 1.0

import math
for nome, peso in (
    ("COMPLETENESS", DQ_SCORE_COMPLETENESS_WEIGHT),
    ("TEMPORAL", DQ_SCORE_TEMPORAL_WEIGHT)
):
    if isinstance(peso, bool) or not isinstance(peso, (int, float)):
        raise TypeError(f"Peso {nome} deve ser numérico.")
    if not math.isfinite(float(peso)) or peso <= 0:
        raise ValueError(f"Peso {nome} deve ser positivo e finito.")

if not isinstance(DQ_SCORE_EXCLUDED_COLUMNS, (list, tuple, set)):
    raise TypeError("DQ_SCORE_EXCLUDED_COLUMNS deve ser lista, tupla ou conjunto.")

In [0]:
# ============================================================
# 10.4 CONTRATO E FUNÇÕES COMPARTILHADAS
# ============================================================

# O QUE FAZ:
# Valida a origem dos indicadores e as funções do módulo 00.

# COMO FAZ:
# Verifica campos, tipos e dependências antes de construir métricas.

# POR QUE É IMPORTANTE:
# Impede que alterações entre módulos produzam resultados silenciosamente incorretos.

# PERGUNTA RESPONDIDA:
# "As entradas seguem o contrato necessário?"

if "schema_profile_df" not in globals() or schema_profile_df is None:
    raise ValueError("Execute 03_SCHEMA_PROFILE antes do Score.")

required_schema = {"coluna", "pct_preenchimento"}
if DQ_SCORE_USE_EFFECTIVE_COMPLETENESS:
    required_schema.add("pct_completude_efetiva")
missing = required_schema - set(schema_profile_df.columns)
if missing:
    raise ValueError(f"Schema Profile sem campos obrigatórios: {sorted(missing)}")

required_functions = (
    "dq_score_metric_schema", "dq_score_pct", "dq_score_aggregate"
)
missing_functions = [
    name for name in required_functions if not callable(globals().get(name))
]
if missing_functions:
    raise ValueError(
        "Adicione as funções do módulo 10 ao 00_FRAMEWORK_CONFIG: "
        + ", ".join(missing_functions)
    )

schema_metricas = dq_score_metric_schema()
expected_fields = [
    "dimensao", "coluna", "metrica", "valor_observado",
    "score", "peso", "elegivel", "natureza", "origem", "justificativa"
]
if schema_metricas.fieldNames() != expected_fields:
    raise ValueError("Schema de métricas do módulo 00 incompatível.")

completeness_field = (
    "pct_completude_efetiva"
    if DQ_SCORE_USE_EFFECTIVE_COMPLETENESS
    else "pct_preenchimento"
)

In [0]:
# ============================================================
# 10.5 MÉTRICA DE COMPLETUDE
# ============================================================

# O QUE FAZ:
# Converte completude observada em indicador operacional por coluna.

# COMO FAZ:
# Projeta os percentuais do Schema Profile sem coletar dados brutos.

# POR QUE É IMPORTANTE:
# Separa presença observada de obrigatoriedade semântica de preenchimento.

# PERGUNTA RESPONDIDA:
# "Qual é a completude operacional das colunas elegíveis?"


schema_cols = schema_profile_df.where(
    ~F.col("coluna").isin(list(DQ_SCORE_EXCLUDED_COLUMNS))
)

score_completude_df = (
    schema_cols.select(
        F.lit("SCHEMA").alias("dimensao"),
        F.col("coluna").cast("string").alias("coluna"),
        F.lit("COMPLETUDE_OPERACIONAL").alias("metrica"),
        F.col(completeness_field).cast("double").alias("valor_observado"),
        dq_score_pct(F.col(completeness_field)).alias("score"),
        F.lit(float(DQ_SCORE_COMPLETENESS_WEIGHT)).alias("peso"),
        F.lit(bool(DQ_SCORE_ENABLE_COMPLETENESS)).alias("elegivel"),
        F.lit("PROXY_OPERACIONAL").alias("natureza"),
        F.lit("03_SCHEMA_PROFILE").alias("origem"),
        F.lit(
            "Completude observada, sem pressupor obrigatoriedade de preenchimento."
        ).alias("justificativa")
    )
    .select(*schema_metricas.fieldNames())
)

In [0]:
# ============================================================
# 10.6 REGRAS TEMPORAIS EXPLÍCITAS
# ============================================================

# O QUE FAZ:
# Pontua somente regras temporais declaradas e comparáveis.

# COMO FAZ:
# Converte percentuais válidos de violação em conformidade.

# POR QUE É IMPORTANTE:
# Não trata hipóteses exploratórias como falhas confirmadas.

# PERGUNTA RESPONDIDA:
# "Qual é a conformidade das regras temporais explicitamente ativadas?"


if DQ_SCORE_ENABLE_TEMPORAL_RULES:
    if (
        "consistency_temporal_profile_df" not in globals()
        or consistency_temporal_profile_df is None
    ):
        raise ValueError(
            "Execute 07_CONSISTENCY_PROFILE no mesmo contexto "
            "antes de habilitar regras temporais."
        )

    temporal_required = {
        "inicio", "fim", "registros_comparaveis",
        "violacoes_pct", "classificacao"
    }
    missing_temporal = temporal_required - set(
        consistency_temporal_profile_df.columns
    )
    if missing_temporal:
        raise ValueError(
            f"Contrato temporal incompatível: {sorted(missing_temporal)}"
        )

    temporal_pct = dq_score_pct(F.col("violacoes_pct"))
    temporal_applicable = (
        (F.col("classificacao") == F.lit("REGRA_EXPLICITA"))
        & (F.col("registros_comparaveis") > 0)
        & temporal_pct.isNotNull()
    )

    score_temporal_df = (
        consistency_temporal_profile_df.select(
            F.lit("CONSISTENCY").alias("dimensao"),
            F.concat_ws(" → ", F.col("inicio"), F.col("fim")).alias("coluna"),
            F.lit("CONFORMIDADE_TEMPORAL").alias("metrica"),
            F.col("violacoes_pct").cast("double").alias("valor_observado"),
            (F.lit(100.0) - temporal_pct).alias("score"),
            F.lit(float(DQ_SCORE_TEMPORAL_WEIGHT)).alias("peso"),
            temporal_applicable.alias("elegivel"),
            F.lit("REGRA_EXPLICITA").alias("natureza"),
            F.lit("07_CONSISTENCY_PROFILE").alias("origem"),
            F.lit(
                "Regra temporal declarada; score = 100 - percentual de violações."
            ).alias("justificativa")
        )
        .select(*schema_metricas.fieldNames())
    )
else:
    score_temporal_df = spark.createDataFrame([], schema_metricas)

In [0]:
# ============================================================
# 10.7 CATÁLOGO DE MÉTRICAS
# ============================================================

# O QUE FAZ:
# Unifica métricas pontuáveis de diferentes dimensões.

# COMO FAZ:
# Combina DataFrames tipados e aplica validação final de elegibilidade.

# POR QUE É IMPORTANTE:
# Preserva valores inválidos como não elegíveis, sem convertê-los em aprovação.

# PERGUNTA RESPONDIDA:
# "Quais métricas estão catalogadas e quais são elegíveis?"


dq_score_metrics_df = (
    score_completude_df
    .unionByName(score_temporal_df)
    .withColumn(
        "elegivel",
        F.coalesce(F.col("elegivel"), F.lit(False))
        & F.col("score").isNotNull()
        & (~F.isnan(F.col("score")))
        & (F.abs(F.col("score")) != F.lit(float("inf")))
        & F.col("score").between(0.0, 100.0)
        & F.col("peso").isNotNull()
        & (F.col("peso") > 0)
        & (~F.isnan(F.col("peso")))
        & (F.abs(F.col("peso")) != F.lit(float("inf")))
    )
)
if DQ_SCORE_DISPLAY_RESULTS:
    display(dq_score_metrics_df)

In [0]:
# ============================================================
# 10.8 SCORE POR DIMENSÃO
# ============================================================

# O QUE FAZ:
# Calcula a média ponderada dos critérios elegíveis por dimensão.

# COMO FAZ:
# Utiliza a função compartilhada de agregação.

# POR QUE É IMPORTANTE:
# Exibe cobertura e distingue dimensão sem avaliação de dimensão aprovada.

# PERGUNTA RESPONDIDA:
# "Qual é a pontuação por dimensão?"


dq_score_dimension_df = dq_score_aggregate(
    dq_score_metrics_df, ["dimensao"]
)
if DQ_SCORE_DISPLAY_RESULTS:
    display(dq_score_dimension_df)

In [0]:
# ============================================================
# 10.9 SCORE POR COLUNA
# ============================================================

# O QUE FAZ:
# Calcula a visão complementar de score por atributo ou relação.

# COMO FAZ:
# Agrupa as métricas elegíveis por identificador de coluna.

# POR QUE É IMPORTANTE:
# Permite investigar resultados sem substituir a visão por dimensão.

# PERGUNTA RESPONDIDA:
# "Quais atributos ou relações merecem investigação?"


dq_score_column_df = dq_score_aggregate(
    dq_score_metrics_df.where(F.col("coluna").isNotNull()),
    ["coluna"]
)
if DQ_SCORE_DISPLAY_RESULTS:
    display(dq_score_column_df)

In [0]:
# ============================================================
# 10.10 SCORE GERAL DO DATASET
# ============================================================

# O QUE FAZ:
# Consolida scores de dimensões com cobertura separada.

# COMO FAZ:
# Calcula média das dimensões pontuadas e proporções de métricas e dimensões.

# POR QUE É IMPORTANTE:
# Evita que dimensões com muitas colunas dominem a pontuação final.

# PERGUNTA RESPONDIDA:
# "Qual é a síntese operacional dos critérios habilitados?"


# Cada dimensão elegível participa uma vez no score geral.
# Pesos configurados afetam a média dentro de cada dimensão.
resumo_dimensoes = dq_score_dimension_df.agg(
    F.count(F.lit(1)).cast("long").alias("dimensoes_catalogadas"),
    F.coalesce(
        F.sum(F.when(F.col("score").isNotNull(), 1).otherwise(0)),
        F.lit(0)
    ).cast("long").alias("dimensoes_pontuadas"),
    F.avg("score").alias("score_geral")
)

resumo_metricas = dq_score_metrics_df.agg(
    F.count(F.lit(1)).cast("long").alias("metricas_catalogadas"),
    F.coalesce(
        F.sum(F.when(F.col("elegivel"), 1).otherwise(0)),
        F.lit(0)
    ).cast("long").alias("metricas_pontuadas")
)

dq_score_dataset_df = (
    resumo_dimensoes.crossJoin(resumo_metricas)
    .withColumn("score_geral", F.round("score_geral", 2))
    .withColumn(
        "cobertura_metricas_pct",
        F.when(
            F.col("metricas_catalogadas") > 0,
            F.round(
                F.col("metricas_pontuadas") * 100.0
                / F.col("metricas_catalogadas"), 2
            )
        )
    )
    .withColumn(
        "cobertura_dimensoes_pct",
        F.when(
            F.col("dimensoes_catalogadas") > 0,
            F.round(
                F.col("dimensoes_pontuadas") * 100.0
                / F.col("dimensoes_catalogadas"), 2
            )
        )
    )
    .withColumn(
        "interpretacao",
        F.lit(
            "Síntese operacional dos critérios habilitados; "
            "não certifica qualidade global."
        )
    )
)
if DQ_SCORE_DISPLAY_RESULTS:
    display(dq_score_dataset_df)

In [0]:
# ============================================================
# 10.11 EVIDÊNCIAS EXPLORATÓRIAS
# ============================================================

# O QUE FAZ:
# Documenta dimensões não penalizadas automaticamente.

# COMO FAZ:
# Cria catálogo tipado de justificativas interpretativas.

# POR QUE É IMPORTANTE:
# Impede que anomalias exploratórias se tornem erros sem regra declarada.

# PERGUNTA RESPONDIDA:
# "Quais evidências não participam automaticamente do Score?"


evidence_rows = [
    ("DISTRIBUTION", "Concentração e distribuição não são erros por si só."),
    ("PATTERN", "Desvios de padrão exigem interpretação semântica."),
    ("DUPLICITY", "Duplicidade pode ser legítima sem definição de chave."),
    ("CONSISTENCY", "Sinais intrínsecos e dependências funcionais candidatas são exploratórios."),
    ("OUTLIER", "Outliers estatísticos não equivalem a registros inválidos."),
    ("CORRELATION", "Correlação não demonstra causalidade ou baixa qualidade.")
]

dq_score_evidence_df = (
    spark.createDataFrame(
        evidence_rows,
        "dimensao STRING, justificativa STRING"
    )
    .withColumn("pontuacao_automatica", F.lit(False))
)
if DQ_SCORE_DISPLAY_RESULTS:
    display(dq_score_evidence_df)

In [0]:
# ============================================================
# 10.12 VALIDAÇÕES OPCIONAIS
# ============================================================

# O QUE FAZ:
# Verifica intervalos, elegibilidade e agregação matemática.

# COMO FAZ:
# Executa testes sintéticos e validações distribuídas quando habilitado.

# POR QUE É IMPORTANTE:
# Detecta regressões sem acrescentar ações Spark à execução padrão.

# PERGUNTA RESPONDIDA:
# "O Score respeita os contratos matemáticos definidos?"


if DQ_SCORE_RUN_TESTS:
    invalidos = dq_score_metrics_df.where(
        F.col("elegivel")
        & (
            F.col("score").isNull()
            | F.isnan(F.col("score"))
            | (F.abs(F.col("score")) == F.lit(float("inf")))
            | (~F.col("score").between(0.0, 100.0))
            | F.col("peso").isNull()
            | (F.col("peso") <= 0)
        )
    ).limit(1).count()
    assert invalidos == 0, "Métricas elegíveis contêm scores ou pesos inválidos."

    sintetico = spark.createDataFrame([
        ("SCHEMA", "a", "TESTE", 90.0, 90.0, 1.0, True, "TESTE", "TESTE", "TESTE"),
        ("SCHEMA", "b", "TESTE", 50.0, 50.0, 1.0, True, "TESTE", "TESTE", "TESTE"),
    ], schema_metricas)

    score_sintetico = dq_score_aggregate(
        sintetico, ["dimensao"]
    ).first()["score"]
    assert abs(score_sintetico - 70.0) < 1e-9

    resumo = dq_score_dataset_df.first()
    assert (
        resumo["score_geral"] is None
        or 0.0 <= resumo["score_geral"] <= 100.0
    )
    print("Testes sintéticos e validações do Score aprovados.")
else:
    print("Testes opcionais desabilitados (DQ_SCORE_RUN_TESTS=False).")